In [1]:
import duckdb
import pandas as pd

from analytics.checks import table_row_counts
from analytics.db import query
from analytics.paths import RAW_OLIST

In [2]:
RAW_OLIST.exists(), sorted(p.name for p in RAW_OLIST.glob("*.csv"))

(True,
 ['olist_customers_dataset.csv',
  'olist_geolocation_dataset.csv',
  'olist_order_items_dataset.csv',
  'olist_order_payments_dataset.csv',
  'olist_order_reviews_dataset.csv',
  'olist_orders_dataset.csv',
  'olist_products_dataset.csv',
  'olist_sellers_dataset.csv',
  'product_category_name_translation.csv'])

In [3]:
query("SELECT version()")

,version
0,"PostgreSQL 18.6 on x86_64-windows, compiled by..."


In [4]:
customers = pd.read_csv(RAW_OLIST / "olist_customers_dataset.csv")
customers["customer_zip_code_prefix"].head()

0    14409
1     9790
2     1151
3     8775
4    13056
Name: customer_zip_code_prefix, dtype: int64

In [5]:
duckdb.sql(f"""
       SELECT COUNT(*)
       FROM read_csv('{(RAW_OLIST / "olist_customers_dataset.csv").as_posix()}', all_varchar = true)
       WHERE customer_zip_code_prefix LIKE '0%'
   """)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        23995 │
└──────────────┘

23,995 customers have a zip code that starts with 0.

In [6]:
query("""
    SELECT 'customers' AS table_name, COUNT(*) AS rows FROM customers
    UNION ALL SELECT 'geolocation', COUNT(*) FROM geolocation
    UNION ALL SELECT 'orders', COUNT(*) FROM orders
    UNION ALL SELECT 'order_items', COUNT(*) FROM order_items
    UNION ALL SELECT 'order_payments', COUNT(*) FROM order_payments
    UNION ALL SELECT 'order_reviews', COUNT(*) FROM order_reviews
    UNION ALL SELECT 'products', COUNT(*) FROM products
    UNION ALL SELECT 'sellers', COUNT(*) FROM sellers
    UNION ALL SELECT 'category_translation', COUNT(*) FROM category_translation
""")

,table_name,rows
0,category_translation,71
1,sellers,3095
2,products,32951
3,order_payments,103886
4,customers,99441
5,orders,99441
6,order_reviews,99224
7,order_items,112650
8,geolocation,1000163


In [7]:
query("SELECT COUNT(*) FROM customers WHERE customer_zip_code_prefix LIKE '0%'")

,count
0,23995


In [8]:
table_row_counts()

,table,rows
0,customers,99441
1,geolocation,1000163
2,orders,99441
3,order_items,112650
4,order_payments,103886
5,order_reviews,99224
6,products,32951
7,sellers,3095
8,category_translation,71


In [9]:
query("""
       SELECT to_char(order_purchase_timestamp, 'YYYY-MM') AS month, COUNT(*) AS orders
       FROM orders
       GROUP BY month
       ORDER BY month
   """)

,month,orders
0,2016-09,4
1,2016-10,324
2,2016-12,1
3,2017-01,800
4,2017-02,1780
5,2017-03,2682
6,2017-04,2404
7,2017-05,3700
8,2017-06,3245
9,2017-07,4026
